# 01 - Data Cleaning

Goal: turn the raw CVM DFP files into a clean table with the key accounts
for the selected retailers, one row per company and year.

**Part 3.1 - Find the CVM code of each company**

We filter companies by their **CVM code** (`CD_CVM`), not by name:
names change over time and vary in spelling, while the code is fixed and unique.

In [ ]:
import pandas as pd

RAW_DIR = "../data/raw"  # the notebook runs from inside notebooks/

## Read the company index

`dfp_cia_aberta_<year>.csv` lists every company that filed a DFP that year.
CVM files use `;` as separator and `latin-1` encoding.

In [ ]:
index_2025 = pd.read_csv(f"{RAW_DIR}/dfp_cia_aberta_2025.csv", sep=";", encoding="latin-1")

print(index_2025.shape)
index_2025.head()

In [ ]:
index_2025.columns

## Search the retailers by name

`str.contains` returns a True/False column (a *boolean mask*).
Using the mask inside `[]` keeps only the rows where it is True.
The `|` inside the pattern means "or", so we search the four names at once.

In [ ]:
pattern = "MAGAZINE LUIZA|RENNER|SBF|C&A"
mask = index_2025["DENOM_CIA"].str.contains(pattern, case=False)

index_2025[mask][["CD_CVM", "DENOM_CIA", "CNPJ_CIA", "VERSAO", "DT_RECEB"]]

## Two things to notice

**1. Two "SBF" companies**

- `24694` - **GRUPO SBF S.A.**: the listed holding company (ticker SBFG3).
- `25410` - **SBF COMÉRCIO DE PRODUTOS ESPORTIVOS S.A.**: an operating subsidiary,
  which also has its own CVM registration.

We use **24694**. The holding's *consolidated* statements already include its
subsidiaries, so adding the subsidiary would count the same revenue twice.

**2. Lojas Renner appears twice with the same code**

The rows differ in `VERSAO` (1 and 2): the company re-filed (*reapresentação*)
its DFP. The most recent version is the valid one. We check below whether the
financial statement files also contain both versions.

In [ ]:
COMPANIES = {
    22470: "Magazine Luiza",
    8133: "Lojas Renner",
    24694: "Grupo SBF",
    24848: "C&A",
}

## Check: are all companies present in every year?

If a company is missing in some year, our 5-year comparison would have a gap.
`isin` checks whether each value belongs to a list.

In [ ]:
for year in range(2021, 2026):
    index_year = pd.read_csv(f"{RAW_DIR}/dfp_cia_aberta_{year}.csv", sep=";", encoding="latin-1")
    found = index_year[index_year["CD_CVM"].isin(COMPANIES.keys())]["CD_CVM"].unique()
    print(year, "found", len(found), "of", len(COMPANIES))

## Check: how many versions are in the financial statement files?

`groupby("CD_CVM")` splits the rows by company, and `nunique()` counts the
distinct values of `VERSAO` in each group. If every count is 1, the file
already keeps only the latest version and we don't need to handle it.

In [ ]:
for year in range(2021, 2026):
    dre = pd.read_csv(f"{RAW_DIR}/dfp_cia_aberta_DRE_con_{year}.csv", sep=";", encoding="latin-1")
    dre = dre[dre["CD_CVM"].isin(COMPANIES.keys())]
    print(year, dre.groupby("CD_CVM")["VERSAO"].nunique().to_dict())

**Conclusion of 3.1:** the four CVM codes are stored in `COMPANIES`.
All companies are present in every year, and the statement files keep
a single (latest) version per company.

---
**Part 3.2 - Load the statements and apply the filters**

We need three statements, all **consolidated** (`_con` files):

| File | Statement |
|---|---|
| `BPA` | Balance sheet - assets |
| `BPP` | Balance sheet - liabilities and equity |
| `DRE` | Income statement |

Filters:
1. Only our four companies (`CD_CVM`).
2. Only the current fiscal year: `ORDEM_EXERC == "ÚLTIMO"`.

Each DFP also brings the **prior year** as a comparative (`PENÚLTIMO`).
If we kept both, every year would appear twice: once as `ÚLTIMO` in its own
file and once as `PENÚLTIMO` in the next year's file.

## Look at one file first

Before writing a loop, it is good practice to open one file and understand it.

In [ ]:
dre_2025 = pd.read_csv(f"{RAW_DIR}/dfp_cia_aberta_DRE_con_2025.csv", sep=";", encoding="latin-1")

print(dre_2025.shape)
dre_2025.head()

In [ ]:
# How many rows of each fiscal year order are in the file?
dre_2025["ORDEM_EXERC"].value_counts()

In [ ]:
# One company, current year only: this is the income statement as published
magalu_2025 = dre_2025[
    (dre_2025["CD_CVM"] == 22470) & (dre_2025["ORDEM_EXERC"] == "ÚLTIMO")
]

magalu_2025[["CD_CONTA", "DS_CONTA", "VL_CONTA", "ESCALA_MOEDA"]]

## A function to load and filter one statement for one year

The same steps repeat for 3 statements x 5 years = 15 files,
so we write them once inside a function.

In [ ]:
def load_statement(statement, year):
    """Read one consolidated statement file and keep our companies, current year only."""
    path = f"{RAW_DIR}/dfp_cia_aberta_{statement}_con_{year}.csv"
    df = pd.read_csv(path, sep=";", encoding="latin-1")

    is_our_company = df["CD_CVM"].isin(COMPANIES.keys())
    is_current_year = df["ORDEM_EXERC"] == "ÚLTIMO"
    df = df[is_our_company & is_current_year].copy()

    df["STATEMENT"] = statement
    df["YEAR"] = year
    return df

In [ ]:
STATEMENTS = ["BPA", "BPP", "DRE"]
YEARS = range(2021, 2026)

frames = []
for statement in STATEMENTS:
    for year in YEARS:
        frames.append(load_statement(statement, year))

statements = pd.concat(frames, ignore_index=True)

print(statements.shape)
statements.head()

## Checks

1. Every company should have rows for every statement and year (no gaps).
2. The fiscal year should end on 31 December for every company and year,
   otherwise the years would not be comparable.

In [ ]:
# Rows per company, statement and year. A zero would mean a gap.
statements.groupby(["CD_CVM", "STATEMENT", "YEAR"]).size().unstack("YEAR")

In [ ]:
# Fiscal year end date for each company and year
statements.groupby(["CD_CVM", "YEAR"])["DT_FIM_EXERC"].unique().unstack("YEAR")

**Conclusion of 3.2:** `statements` holds the consolidated BPA, BPP and DRE of the
four companies for 2021-2025, current fiscal year only, with no gaps.

---
**Part 3.3 - Convert all values to reais**

`ESCALA_MOEDA` tells the unit of `VL_CONTA`:
- `MIL`: thousands of reais
- `UNIDADE`: reais

We create a column `VALUE` with every amount in reais, so companies that
report in different scales can be compared directly.

In [ ]:
# Which scales appear in our data?
statements["ESCALA_MOEDA"].value_counts()

In [ ]:
SCALE_FACTOR = {"UNIDADE": 1, "MIL": 1000}

statements["SCALE_FACTOR"] = statements["ESCALA_MOEDA"].map(SCALE_FACTOR)

# Fail fast: a scale that is not in the dictionary becomes NaN after map()
if statements["SCALE_FACTOR"].isna().any():
    unknown = statements[statements["SCALE_FACTOR"].isna()]["ESCALA_MOEDA"].unique()
    raise ValueError(f"Unknown scale(s) in ESCALA_MOEDA: {unknown}")

statements["VALUE"] = statements["VL_CONTA"] * statements["SCALE_FACTOR"]

In [ ]:
# Sanity check: Magazine Luiza net revenue (account 3.01) in 2025
check = statements[
    (statements["CD_CVM"] == 22470)
    & (statements["YEAR"] == 2025)
    & (statements["CD_CONTA"] == "3.01")
]

check[["DS_CONTA", "VL_CONTA", "ESCALA_MOEDA", "VALUE"]]

**Conclusion of 3.3:** `VALUE` holds every amount in reais.
From now on we use `VALUE` instead of `VL_CONTA`.

---
**Part 3.4 - Map the accounts and build the final table**

We only use **fixed accounts** (`ST_CONTA_FIXA == "S"`): they belong to the
standard CVM chart of accounts, so the same code means the same thing for
every company. Company-specific sub-accounts (`"N"`) are not comparable.

| Code | CVM description | Our name |
|---|---|---|
| `1` | Ativo Total | `total_assets` |
| `1.01` | Ativo Circulante | `current_assets` |
| `2.01` | Passivo Circulante | `current_liabilities` |
| `2.02` | Passivo Não Circulante | `noncurrent_liabilities` |
| `2.03` | Patrimônio Líquido Consolidado | `equity` |
| `3.01` | Receita de Venda de Bens e/ou Serviços | `net_revenue` |
| `3.03` | Resultado Bruto | `gross_profit` |
| `3.05` | Resultado Antes do Resultado Financeiro e dos Tributos | `operating_income` |
| `3.11` | Lucro/Prejuízo Consolidado do Período | `net_income` |

Notes:
- `2.02` is not in the original list, but we need it for the debt ratio in step 4
  (total liabilities = current + non-current).
- `2.03` and `3.11` both **include non-controlling interests**, so ROE
  (net income / equity) compares two consistent figures.

In [ ]:
ACCOUNTS = {
    "1": "total_assets",
    "1.01": "current_assets",
    "2.01": "current_liabilities",
    "2.02": "noncurrent_liabilities",
    "2.03": "equity",
    "3.01": "net_revenue",
    "3.03": "gross_profit",
    "3.05": "operating_income",
    "3.11": "net_income",
}

selected = statements[statements["CD_CONTA"].isin(ACCOUNTS.keys())].copy()
selected["ACCOUNT"] = selected["CD_CONTA"].map(ACCOUNTS)

selected[["CD_CVM", "YEAR", "CD_CONTA", "DS_CONTA", "ACCOUNT", "VALUE"]].head(10)

## Checks before reshaping

1. Every selected account must be a fixed account.
2. Every company and year must have all 9 accounts, no more, no less.

In [ ]:
if not (selected["ST_CONTA_FIXA"] == "S").all():
    raise ValueError("Some selected accounts are not fixed CVM accounts")

accounts_per_company_year = selected.groupby(["CD_CVM", "YEAR"]).size()
if not (accounts_per_company_year == len(ACCOUNTS)).all():
    raise ValueError("Some company/year is missing accounts or has duplicates")

print("OK:", len(accounts_per_company_year), "company-years with", len(ACCOUNTS), "accounts each")

## Reshape: one row per company and year

Right now the data is in **long** format: one row per account.
For the ratios we want **wide** format: one row per company and year,
one column per account. `pivot` does this reshape.

In [ ]:
financials = selected.pivot(index=["CD_CVM", "YEAR"], columns="ACCOUNT", values="VALUE")
financials = financials.reset_index()

financials["COMPANY"] = financials["CD_CVM"].map(COMPANIES)

columns_order = ["COMPANY", "CD_CVM", "YEAR"] + list(ACCOUNTS.values())
financials = financials[columns_order]
financials.columns.name = None  # remove the leftover "ACCOUNT" label

financials

## Accounting check: Assets = Liabilities + Equity

If the mapping is right, the balance sheet must balance for every row.

In [ ]:
liabilities_and_equity = (
    financials["current_liabilities"]
    + financials["noncurrent_liabilities"]
    + financials["equity"]
)
difference = financials["total_assets"] - liabilities_and_equity

print("Largest difference (R$):", difference.abs().max())

## Easier to read: values in R$ millions

In [ ]:
value_columns = list(ACCOUNTS.values())

in_millions = financials.copy()
in_millions[value_columns] = (in_millions[value_columns] / 1_000_000).round(0)

in_millions

## Save the clean table

`data/processed/` is ignored by Git, like `data/raw/`: anyone can rebuild
this file by running the download script and this notebook.

In [ ]:
financials.to_csv("../data/processed/financials.csv", index=False)
print("Saved", financials.shape[0], "rows")

**Conclusion of 3.4:** `financials` has one row per company and year (4 x 5 = 20 rows),
with the 9 accounts in reais, ready for the ratios in step 4.